# WAI illustrious SDXL v1.5 — Colab Studio ⚡

ดาวน์โหลดและรันโมเดลสาธารณะจาก Hugging Face: `dhead/waiIllustriousSDXL_v150`

## เริ่มใช้งาน
1. เลือก **Runtime → Change runtime type → T4 GPU**
2. รันเซลล์ติดตั้งแพ็กเกจ; หากมีข้อความขอรีสตาร์ต runtime ให้รีสตาร์ตแล้วรันต่อจากเซลล์ตรวจ GPU
3. รันเซลล์โหลดโมเดลจาก Hugging Face ไม่ต้องใช้ Hugging Face token
4. เปิด UI ที่แสดงใน Colab, ใส่ prompt และกด **สร้างภาพ**

> ใช้เฉพาะภาพตัวละครผู้ใหญ่ที่สมมติขึ้นและอยู่ภายใต้กฎหมายของพื้นที่และข้อกำหนดของ Hugging Face/Colab ไม่แนะนำให้ใช้ Prompt ที่เกี่ยวข้องกับบุคคลเยาว์หรือตัวตนจริงโดยไม่ได้รับอนุญาต

In [ ]:
# ติดตั้งแพ็กเกจ; ไม่แก้หรือเขียนทับ PyTorch/CUDA ของ Colab
!pip -q install -U diffusers transformers accelerate safetensors gradio huggingface_hub

In [ ]:
import gc
import os
import random
import torch
import gradio as gr
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

if not torch.cuda.is_available():
    raise RuntimeError('ไม่พบ GPU: เลือก Runtime → Change runtime type → T4 GPU แล้วรันใหม่')

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {gpu_name} | VRAM: {gpu_memory_gb:.1f} GB')

MODEL_ID = 'dhead/waiIllustriousSDXL_v150'
CACHE_DIR = '/content/huggingface-cache'
pipe = None

In [ ]:
def load_model():
    global pipe
    if pipe is not None:
        del pipe
        pipe = None
        gc.collect()
        torch.cuda.empty_cache()

    print(f'กำลังดาวน์โหลด/โหลด {MODEL_ID} จาก Hugging Face — ไม่ใช้ token')
    new_pipe = StableDiffusionXLPipeline.from_pretrained(
        MODEL_ID,
        cache_dir=CACHE_DIR,
        torch_dtype=torch.float16,
        use_safetensors=True,
    )
    new_pipe.scheduler = DPMSolverMultistepScheduler.from_config(
        new_pipe.scheduler.config,
        use_karras_sigmas=True,
    )
    new_pipe.enable_vae_slicing()
    new_pipe.enable_vae_tiling()
    new_pipe.enable_model_cpu_offload()
    new_pipe.set_progress_bar_config(disable=True)
    pipe = new_pipe
    del new_pipe
    torch.cuda.empty_cache()
    return 'พร้อมใช้งาน: dhead/waiIllustriousSDXL_v150 (ไม่ใช้ token)'

status = load_model()
print(status)

In [ ]:
def generate(prompt, negative_prompt, width, height, steps, cfg, seed):
    if pipe is None:
        raise gr.Error('ยังไม่ได้โหลดโมเดล')
    if not prompt or not prompt.strip():
        raise gr.Error('กรุณากรอก Prompt')

    seed = int(seed)
    if seed == -1:
        seed = random.randint(0, 2**31 - 1)
    elif seed < 0:
        raise gr.Error('Seed ต้องเป็น -1 (สุ่ม) หรือเลขตั้งแต่ 0 ขึ้นไป')

    try:
        generator = torch.Generator(device='cuda').manual_seed(seed)
        result = pipe(
            prompt=prompt.strip(),
            negative_prompt=negative_prompt.strip() or None,
            width=int(width),
            height=int(height),
            num_inference_steps=int(steps),
            guidance_scale=float(cfg),
            num_images_per_prompt=1,
            generator=generator,
        )
        images = result.images
        del result
        torch.cuda.empty_cache()
        return images, f'Seed: {seed}'
    except Exception as exc:
        gc.collect()
        torch.cuda.empty_cache()
        message = str(exc)
        if 'out of memory' in message.lower():
            message += ' — ลดขนาดเป็น 768×768 หรือ 640×640 แล้วลองใหม่'
        raise gr.Error(f'สร้างภาพไม่สำเร็จ: {message}')

sizes = [640, 768, 832, 896, 1024]

with gr.Blocks(theme=gr.themes.Soft(), title='WAI illustrious SDXL v1.5') as demo:
    gr.Markdown('# WAI illustrious SDXL v1.5 ⚡\nโหลดจาก Hugging Face แบบไม่ใช้ token')
    with gr.Row():
        with gr.Column():
            prompt = gr.Textbox(label='Prompt', lines=5, placeholder='อธิบายภาพที่ต้องการ')
            negative = gr.Textbox(label='Negative prompt', lines=3, placeholder='สิ่งที่ไม่ต้องการในภาพ')
            with gr.Row():
                width = gr.Dropdown(sizes, value=768, label='กว้าง')
                height = gr.Dropdown(sizes, value=768, label='สูง')
            with gr.Row():
                steps = gr.Slider(10, 40, value=24, step=1, label='Steps')
                cfg = gr.Slider(1, 10, value=5, step=0.5, label='CFG Scale')
            seed = gr.Number(value=-1, precision=0, label='Seed (-1 = สุ่ม)')
            button = gr.Button('สร้างภาพ', variant='primary')
        with gr.Column():
            gallery = gr.Gallery(label='ภาพที่สร้าง', columns=1, height='auto')
            seed_output = gr.Textbox(label='Seed ที่ใช้', interactive=False)

    button.click(generate, inputs=[prompt, negative, width, height, steps, cfg, seed], outputs=[gallery, seed_output])

demo.queue().launch(share=False, inline=True)